In [1]:
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install pandas numpy tables matplotlib --quiet


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\Arash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\Arash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\Arash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, BatchNorm
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import zipfile

# Set device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
def unzip_data(zip_path='METR-LA.zip', data_dir='METR-LA'):
    if not os.path.exists(data_dir):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
    print(f"Data directory: {data_dir}")

In [4]:
def load_metr_la_data(data_dir='METR-LA'):
    adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))  # Shape: [207, 207]
    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))  # Shape: [34272, 207, 2]

    # Handle 3D node_values (select speed feature and transpose)
    print(f"Original node_values shape: {node_values.shape}")
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]  # Shape: [34272, 207]
        node_values = node_values.T  # Shape: [207, 34272]
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")

    print(f"Processed node_values shape: {node_values.shape}")

    # Normalize node values
    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T  # Shape: [207, 34272]

    # Convert adjacency matrix to edge_index and edge_weight
    edge_index = np.where(adj_mat > 0)  # Tuple of NumPy arrays
    edge_index = torch.tensor(np.array(edge_index), dtype=torch.long)  # Convert to NumPy array first
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)
    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)
    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")

    return node_values, edge_index, edge_weight, scaler

In [5]:
def create_data_samples(node_values, in_timestep=12, out_timestep=12):
    features, labels = [], []
    for i in range(node_values.shape[1] - in_timestep - out_timestep + 1):
        features.append(node_values[:, i:i + in_timestep])
        labels.append(node_values[:, i + in_timestep:i + in_timestep + out_timestep])
    features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)
    labels = torch.tensor(labels, dtype=torch.float).permute(0, 2, 1)
    return features, labels

In [6]:
def create_data_list(features, labels, edge_index, walk_length=20):
    transform = AddRandomWalkPE(walk_length=walk_length, attr_name='pe')
    data_list = []
    for i in range(features.shape[0]):
        x = features[i].T
        y = labels[i].T
        data = Data(x=x, y=y, edge_index=edge_index)
        data = transform(data)
        data_list.append(data)
    return data_list

In [7]:
def create_data_loaders(data_list, batch_size=32):
    train_data = data_list[:int(0.7 * len(data_list))]
    val_data = data_list[int(0.7 * len(data_list)):int(0.9 * len(data_list))]
    test_data = data_list[int(0.9 * len(data_list)):]

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}, Test samples: {len(test_data)}")
    return train_loader, val_loader, test_loader

In [8]:
class GraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, pe_dim=20, heads=4):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim)
        self.pe_lin = nn.Linear(pe_dim, pe_dim)
        self.convs = nn.ModuleList([
            TransformerConv(hidden_channels, hidden_channels, heads=heads, concat=False, dropout=0.1)
            for _ in range(num_layers)
        ])
        self.norm = BatchNorm(hidden_channels)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, x, edge_index, pe, batch):
        x = self.node_emb(x)
        pe = self.pe_lin(pe)
        x = torch.cat([x, pe], dim=-1)
        for conv in self.convs:
            x = conv(x, edge_index)
            x = self.norm(x)
            x = torch.relu(x)
        x = self.mlp(x)
        batch_size = batch.max().item() + 1
        num_nodes = x.size(0) // batch_size
        return x.view(batch_size, num_nodes, -1)

In [9]:
def compute_metrics(pred, target, scaler):
    # pred, target shape: [batch_size, num_nodes, out_timestep]
    batch_size, num_nodes, out_timestep = pred.shape
    
    # Reshape for inverse transform: [batch_size * out_timestep, num_nodes]
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)  # [batch_size * out_timestep, 207]
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)  # [batch_size * out_timestep, 207]
    
    # Inverse transform
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    
    # Debug: Print min and max target values
    print(f"Batch min target value (unscaled): {target_unscaled.min().item():.4f} mph")
    print(f"Batch max target value (unscaled): {target_unscaled.max().item():.4f} mph")
    
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0  # Avoid small values
    if mask.sum() > 0:  # Check if mask is non-empty
        mape = torch.mean(torch.abs((pred_unscaled[mask] - target_unscaled[mask]) / (target_unscaled[mask] + 1e-2))) * 100
    else:
        mape = 0.0  # Return 0 if all values are <= 1.0 mph
        print("Warning: MAPE set to 0 because all target values are <= 1.0 mph")
    return mae, rmse, mape.item()

In [10]:
def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5):
    os.makedirs(output_dir, exist_ok=True)

    # pred, target shape: [batch_size, num_nodes, out_timestep]
    # Select the first batch: [num_nodes, out_timestep]
    pred = pred[0].cpu().numpy()  # shape: [207, 12]
    target = target[0].cpu().numpy()  # shape: [207, 12]

    # Inverse transform: expects [n_samples, 207], so transpose to [12, 207]
    pred = scaler.inverse_transform(pred.T).T  # shape: [207, 12]
    target = scaler.inverse_transform(target.T).T  # shape: [207, 12]

    # Plot
    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred.shape[0])):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred[i], label='Predicted', marker='o')
        plt.plot(target[i], label='Actual', marker='x')
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'prediction_plot.png'))
    plt.close()

In [11]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20):
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0
        for data in train_loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.pe, data.batch)
                batch_size = data.batch.max().item() + 1
                num_nodes = out.shape[1]
                target = data.y.view(batch_size, num_nodes, -1)
                loss = criterion(out, target)
                total_val_loss += loss.item()

        val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return model

In [12]:
def evaluate_model(model, test_loader, scaler):
    model.eval()
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    first_batch_data = None
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.pe, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target, scaler)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                first_batch_data = (out, target)  # Store for visualization

    avg_mae = total_mae / num_batches
    avg_rmse = total_rmse / num_batches
    avg_mape = total_mape / num_batches
    print(f"Test MAE: {avg_mae:.4f} mph")
    print(f"Test RMSE: {avg_rmse:.4f} mph")
    print(f"Test MAPE: {avg_mape:.4f}%")

    return avg_mae, avg_rmse, avg_mape, first_batch_data

In [ ]:
# Main workflow
unzip_data()
torch.cuda.empty_cache()
node_values, edge_index, edge_weight, scaler = load_metr_la_data()
features, labels = create_data_samples(node_values)
data_list = create_data_list(features, labels, edge_index)
train_loader, val_loader, test_loader = create_data_loaders(data_list)

model = GraphTransformer(in_channels=12, hidden_channels=64, out_channels=12, num_layers=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.MSELoss()

model = train_model(model, train_loader, val_loader, criterion, optimizer)
mae, rmse, mape, first_batch_data = evaluate_model(model, test_loader, scaler)

if first_batch_data is not None:
    visualize_predictions(first_batch_data[0], first_batch_data[1], scaler)

Data directory: METR-LA
Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 34272)
edge_index device: cpu, shape: torch.Size([2, 1722])
edge_weight device: cpu, shape: torch.Size([1722])


C:\Users\Arash\AppData\Local\Temp\ipykernel_29540\3572316204.py:6: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  features = torch.tensor(features, dtype=torch.float).permute(0, 2, 1)


Train samples: 23974, Val samples: 6850, Test samples: 3425
Epoch 1, Train Loss: 0.0371, Val Loss: 0.1539
Epoch 2, Train Loss: 0.0233, Val Loss: 0.1678
